# 🧪 Exploring Self-Supervised Learning with DINO for Image Classification

## 🎯 1. Objective
Use the DINO pretrained ViT (Vision Transformer) model to extract meaningful image features without labels.

* Perform linear evaluation by training a simple linear classifier on top of frozen features.
* Compare results with a supervised baseline on a small labeled dataset.
* Gain insights into the power of self-supervised representations for downstream tasks.

## 📦 2. Python Dependencies

To run this notebook, make sure you have the following packages installed:

```bash
pip install torch torchvision matplotlib scikit-learn seaborn timm
```

⚠️ If you're using Google Colab or a similar environment, most of these libraries are pre-installed.

## ⚙️ 3. Import Dependencies

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 🗂️ 3. Load and Preprocess Dataset

* Download the CIFAR10 dataset with a few classes
* Resize images to 224×224 to match DINI pretrained input requirements.
* Create a small dataset with only 200 images per class. 


In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 🧪 Load CIFAR-10
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# 🎯 Keep only 3 classes
selected_classes = [0, 1, 2]
selected_indices = [i for i, (_, label) in enumerate(full_dataset) if label in selected_classes]

# 🎯 Limit to 200 samples per class
class_counts = {c: 0 for c in selected_classes}
limited_indices = []

for idx in selected_indices:
    label = full_dataset[idx][1]
    if class_counts[label] < 200:
        limited_indices.append(idx)
        class_counts[label] += 1
    if all(c >= 200 for c in class_counts.values()):
        break

reduced_dataset = Subset(full_dataset, limited_indices)


## ⬇️ 4. Load DINO Pretrained ViT Model

In [ ]:
# Some functions are not available for mps
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

# Load DINO pretrained ViT backend from Facebook Research
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

model.eval().to(device)

## 🧬 5. Extract Features for Dataset

In [ ]:
def extract_features(dataset, model, device, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features = []
    labels = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            feats = model(images)
            features.append(feats.cpu())
            labels.extend(targets.numpy())
    features = torch.cat(features).numpy()
    return features, np.array(labels)

features, labels = extract_features(reduced_dataset, model, device)


## 🏋️‍♂️ 6. Train Logistic Regression on Features

In [ ]:
torch.manual_seed(22)

train_split, val_split = torch.utils.data.random_split(range(len(features)), [0.8, 0.2])

X_train, y_train = features[train_split], labels[train_split]
X_val, y_val = features[val_split], labels[val_split] 

clf = LogisticRegression(max_iter=1000, solver='lbfgs')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)

acc = accuracy_score(y_val, y_pred)
print(f'Linear classifier accuracy on DINO features: {acc:.4f}')


## 🆚 8. Baseline: Train CNN on Same Dataset  

In [ ]:
from torchvision import transforms, models

# Fine tune a resnet18 pretrained on ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the classifier head
num_classes = len(selected_classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Split into train/test

train_dataset = torch.utils.data.Subset(reduced_dataset, train_split)
val_dataset = torch.utils.data.Subset(reduced_dataset, val_split)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train_model(model, loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    print("Training complete.")

train_model(model, train_loader, criterion, optimizer)


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")


evaluate_model(model, val_loader)


## 📊 9. Compare Results

Compare accuracy and training times between:

* Linear classification on DINO features 

* Training CNN

## 🤔 9. Discussion & Questions
1. How does self-supervised pretraining affect classification performance?
2. When would you prefer apply a linear classifier over full CNN training?
3. What are the limitations of self-supervised models like DINO?